# Reliability Diagram — U-Net Calibration

A well-calibrated model should produce probabilities that match observed frequencies: when it predicts p=0.7 for a set of pixels, roughly 70% of those should actually be seabed. The reliability diagram plots this relationship, and the Expected Calibration Error (ECE) summarises the gap between predicted confidence and actual accuracy in a single number.

Since the ground truth is itself algorithmically generated (ensemble of simple + edge detectors), perfect calibration is not expected — but the diagram shows whether the model's raw probabilities are trustworthy or whether post-hoc calibration (e.g. temperature scaling) would help.

In [ ]:
import sys
sys.path.insert(0, '../../..')

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import segmentation_models_pytorch as smp

from deep_learning.unet_model import create_unet
from deep_learning.evaluate import evaluate_file, plot_reliability_diagram
from validation.metrics import calibration_curve

## Load checkpoint and data paths

In [ ]:
REPO_ROOT = Path('../../../..').resolve()
DATASETS = REPO_ROOT / 'Datasets'
ZARR_DIR = DATASETS / 'echopype_zarr'
PATCH_CACHE = DATASETS / 'patch_cache'
CHECKPOINT_DIR = REPO_ROOT / 'code' / 'checkpoints'

# Find the latest checkpoint
checkpoints = sorted(CHECKPOINT_DIR.glob('unet_resnet34_*_best.pth'))
CHECKPOINT = checkpoints[-1] if checkpoints else CHECKPOINT_DIR / 'unet_resnet34_best.pth'

device = 'mps' if torch.backends.mps.is_available() else 'cpu'

ckpt = torch.load(CHECKPOINT, map_location=device, weights_only=False)
cfg = ckpt.get('config', {})

# Reconstruct model (handle scse decoder attention if present)
if cfg.get('decoder_attention') == 'scse':
    model = smp.Unet(
        encoder_name=cfg.get('encoder_name', 'resnet34'),
        encoder_weights=None,
        in_channels=3,
        classes=1,
        activation=None,
        decoder_attention_type='scse',
    )
else:
    model = create_unet(
        encoder_name=cfg.get('encoder_name', 'resnet34'),
        in_channels=3,
        classes=1,
    )

model.load_state_dict(ckpt['model_state_dict'])
model = model.to(device)
model.eval()

target_size = tuple(cfg.get('target_size', (256, 256)))
freq1_hz = cfg.get('freq1_hz', 38_000)
freq2_hz = cfg.get('freq2_hz', 200_000)
vmin = cfg.get('vmin', -80.0)
vmax = cfg.get('vmax', -34.0)
stride = cfg.get('stride', 128)

print(f'Device: {device}')
print(f'Checkpoint: {CHECKPOINT.name}')
print(f'Encoder: {cfg.get("encoder_name", "resnet34")}')
print(f'Target size: {target_size}')
print(f'Loaded from epoch {ckpt.get("epoch", "?")} — val IoU {ckpt.get("val_iou", "?"):.4f}')

In [ ]:
# Determine validation files using the same seed-based split as training
zarr_paths = sorted([
    ZARR_DIR / npz.name.split('_')[0].replace('.npz', '.zarr')
    for npz in PATCH_CACHE.glob('*.npz')
])
zarr_paths = sorted(set(zp for zp in zarr_paths if zp.exists()))
if not zarr_paths:
    zarr_paths = sorted(ZARR_DIR.glob('*.zarr'))

seed = 42
val_fraction = 0.2
n_files = len(zarr_paths)
rng = np.random.RandomState(seed)
indices = rng.permutation(n_files)
n_val = max(1, int(n_files * val_fraction))
val_indices = set(indices[:n_val])
val_zarr = [zarr_paths[i] for i in range(n_files) if i in val_indices]

print(f'Total zarr files: {n_files}')
print(f'Validation files: {len(val_zarr)}')
for vz in val_zarr:
    print(f'  {vz.name}')

## Collect predictions across validation files

Run full-echogram inference on each validation file and flatten all pixel-level predictions and ground truth masks into a single pair of arrays. Aggregating across files gives a more representative calibration picture than any single echogram.

In [ ]:
all_pred_flat = []
all_gt_flat = []
per_file_data = []

for vz in val_zarr:
    print(f'Evaluating {vz.name} ...')
    metrics, prediction, gt_mask, sv_channels, range_arr = evaluate_file(
        model, vz,
        freq1_hz=freq1_hz, freq2_hz=freq2_hz,
        target_size=target_size, device=device,
        stride=stride, vmin=vmin, vmax=vmax,
    )
    print(f'  IoU={metrics.iou:.4f}  Dice={metrics.dice:.4f}  '
          f'Bottom err: {metrics.bottom_error_mean_m:.2f}m')

    all_pred_flat.append(prediction.ravel())
    all_gt_flat.append(gt_mask.ravel())
    per_file_data.append({
        'name': vz.name,
        'prediction': prediction,
        'gt_mask': gt_mask,
        'metrics': metrics,
    })

all_pred = np.concatenate(all_pred_flat)
all_gt = np.concatenate(all_gt_flat)
print(f'\nTotal pixels: {len(all_pred):,}')

## Aggregate reliability diagram

In [ ]:
fig, ece = plot_reliability_diagram(all_pred, all_gt, n_bins=20)
plt.show()
print(f'ECE: {ece:.4f}')
print(f'Avg. confidence: {all_pred.mean():.4f}')
print(f'Avg. accuracy: {all_gt.astype(bool).mean():.4f}')

## Per-file reliability diagrams

Check whether calibration is consistent across files or whether some echograms are significantly better or worse calibrated than others.

In [ ]:
n_files_show = len(per_file_data)
n_cols = min(3, n_files_show)
n_rows = (n_files_show + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 6 * n_rows),
                          squeeze=False)

for idx, fd in enumerate(per_file_data):
    row, col = divmod(idx, n_cols)
    ax = axes[row, col]

    _, obs, mp, counts, file_ece = calibration_curve(
        fd['prediction'], fd['gt_mask'], n_bins=20)

    non_empty = counts > 0
    bp_conf = mp[non_empty]
    bp_acc = obs[non_empty]

    ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, alpha=0.7)
    ax.plot(bp_conf, bp_acc, 'o-', color='#333333', linewidth=2,
            markersize=5, markerfacecolor='#666666',
            markeredgecolor='#333333', markeredgewidth=1.0)

    ax.set_title(f'{fd["name"]}\nECE = {file_ece:.4f}', fontsize=11)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.25)
    ax.set_xlabel('Confidence')
    ax.set_ylabel('Accuracy')

# Hide unused axes
for idx in range(n_files_show, n_rows * n_cols):
    row, col = divmod(idx, n_cols)
    axes[row, col].set_visible(False)

plt.tight_layout()
plt.show()

## Save figure for thesis

In [ ]:
save_path = REPO_ROOT / 'Figure-Creation' / 'reliability_diagram_real.png'
fig, ece = plot_reliability_diagram(
    all_pred, all_gt, n_bins=20,
    save_path=save_path,
)
plt.show()